# World Cup Score Predictor — Clean Feature Engineering Notebook

This notebook builds a clean pre-match feature dataset for the World Cup score predictor.

Main goals:
- One row per match.
- Use only information available before kickoff.
- Convert post-match ELO/rank values into pre-match values.
- Keep a focused, relevant feature set.
- Evaluate with expanding-window World Cup backtesting.

## 1. Imports and Paths

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import exp, factorial

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 120)

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUTS_DIR = BASE_DIR / "outputs" / "evaluation"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)

RAW_DIR: ../data/raw
PROCESSED_DIR: ../data/processed
OUTPUTS_DIR: ../outputs/evaluation


## 2. Load Yearly ELO Match Files

In [3]:
elo_files = sorted(RAW_DIR.glob("elo_*_results.csv"))

if not elo_files:
    raise FileNotFoundError("No files found matching data/raw/elo_*_results.csv")

dfs = []
for file in elo_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    dfs.append(temp)

elo_df = pd.concat(dfs, ignore_index=True)

print("Number of ELO files:", len(elo_files))
print("Loaded ELO shape:", elo_df.shape)
display(elo_df.head())

Number of ELO files: 26
Loaded ELO shape: (24260, 16)


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,rating_b,rank_change_a,rank_change_b,rank_a,rank_b,source_file
0,2001-01-04,Kenya,Zambia,2,1,Friendly,Kenya,11,-11,1339,1444,3,-5,119,91,elo_2001_results.csv
1,2001-01-06,Costa Rica,Guatemala,5,2,World Cup qualifier,the United States,32,-32,1638,1543,5,-6,53,67,elo_2001_results.csv
2,2001-01-06,Egypt,United Arab Emirates,2,1,Friendly,Egypt,4,-4,1686,1542,0,0,42,68,elo_2001_results.csv
3,2001-01-06,Namibia,Lesotho,1,0,Friendly tournament,Swaziland,10,-10,1336,1186,4,-2,120,158,elo_2001_results.csv
4,2001-01-06,Qatar,Jordan,3,1,Friendly,Qatar,10,-10,1505,1459,2,-1,76,81,elo_2001_results.csv


## 3. Basic Cleaning and No-Leakage Pre-Match Features

In [4]:
df = elo_df.copy()

required_cols = [
    "date", "team_a", "team_b", "goals_a", "goals_b", "competition", "location",
    "rating_change_a", "rating_change_b", "rating_a", "rating_b",
    "rank_change_a", "rank_change_b", "rank_a", "rank_b"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "team_a", "team_b", "goals_a", "goals_b"])
df = df.sort_values("date").reset_index(drop=True)

# Targets
df["target_goals_a"] = df["goals_a"].astype(int)
df["target_goals_b"] = df["goals_b"].astype(int)
df["target_goal_diff"] = df["target_goals_a"] - df["target_goals_b"]
df["target_total_goals"] = df["target_goals_a"] + df["target_goals_b"]

# Result points
df["result_a"] = np.where(df["goals_a"] > df["goals_b"], 3,
                  np.where(df["goals_a"] == df["goals_b"], 1, 0))
df["result_b"] = np.where(df["goals_b"] > df["goals_a"], 3,
                  np.where(df["goals_a"] == df["goals_b"], 1, 0))

# Raw rating/rank values are post-match, so reconstruct pre-match values
df["rating_a_before"] = df["rating_a"] - df["rating_change_a"]
df["rating_b_before"] = df["rating_b"] - df["rating_change_b"]

df["rank_a_before"] = df["rank_a"] - df["rank_change_a"]
df["rank_b_before"] = df["rank_b"] - df["rank_change_b"]

df["elo_diff"] = df["rating_a_before"] - df["rating_b_before"]
df["rank_diff"] = df["rank_a_before"] - df["rank_b_before"]

# Home advantage for team_a
df["is_home_adv"] = (
    df["team_a"].astype(str).str.strip().str.lower()
    ==
    df["location"].astype(str).str.strip().str.lower()
).astype(int)

print("Cleaned shape:", df.shape)
display(df[[
    "date", "team_a", "team_b", "competition", "location",
    "rating_a_before", "rating_b_before", "elo_diff",
    "rank_a_before", "rank_b_before", "rank_diff",
    "is_home_adv", "target_goals_a", "target_goals_b"
]].head())

Cleaned shape: (24260, 29)


,date,team_a,team_b,competition,location,rating_a_before,rating_b_before,elo_diff,rank_a_before,rank_b_before,rank_diff,is_home_adv,target_goals_a,target_goals_b
0,2001-01-04,Kenya,Zambia,Friendly,Kenya,1328,1455,-127,116,96,20,1,2,1
1,2001-01-06,Costa Rica,Guatemala,World Cup qualifier,the United States,1606,1575,31,48,73,-25,0,5,2
2,2001-01-06,Egypt,United Arab Emirates,Friendly,Egypt,1682,1546,136,42,68,-26,1,2,1
3,2001-01-06,Namibia,Lesotho,Friendly tournament,Swaziland,1326,1196,130,116,160,-44,0,1,0
4,2001-01-06,Qatar,Jordan,Friendly,Qatar,1495,1469,26,74,82,-8,1,3,1


## 4. Rolling Team Features

In [5]:
WINDOW = 5
team_history = {}

ELO_BASE = df[["rating_a_before", "rating_b_before"]].stack().mean()

features = {
    "team_a_form_last5": [],
    "team_b_form_last5": [],
    "team_a_goals_for_avg_last5": [],
    "team_b_goals_for_avg_last5": [],
    "team_a_goals_against_avg_last5": [],
    "team_b_goals_against_avg_last5": [],
    "team_a_weighted_goals_for_avg_last5": [],
    "team_b_weighted_goals_for_avg_last5": [],
    "team_a_weighted_goals_against_avg_last5": [],
    "team_b_weighted_goals_against_avg_last5": [],
    "team_a_avg_opponent_elo_last5": [],
    "team_b_avg_opponent_elo_last5": [],
    "team_a_rating_change_avg_last5": [],
    "team_b_rating_change_avg_last5": [],
    "team_a_matches_played_before": [],
    "team_b_matches_played_before": [],
    "team_a_days_since_last_match": [],
    "team_b_days_since_last_match": [],
    "team_a_win_streak": [],
    "team_b_win_streak": [],
}

def mean_or_zero(values):
    return float(np.mean(values)) if len(values) > 0 else 0.0

def current_win_streak(history):
    streak = 0
    for match in reversed(history):
        if match["points"] == 3:
            streak += 1
        else:
            break
    return streak

for _, row in df.iterrows():
    team_a = row["team_a"]
    team_b = row["team_b"]
    date = row["date"]

    team_history.setdefault(team_a, [])
    team_history.setdefault(team_b, [])

    hist_a_all = team_history[team_a]
    hist_b_all = team_history[team_b]

    hist_a = hist_a_all[-WINDOW:]
    hist_b = hist_b_all[-WINDOW:]

    features["team_a_form_last5"].append(mean_or_zero([x["points"] for x in hist_a]))
    features["team_b_form_last5"].append(mean_or_zero([x["points"] for x in hist_b]))

    features["team_a_goals_for_avg_last5"].append(mean_or_zero([x["goals_for"] for x in hist_a]))
    features["team_b_goals_for_avg_last5"].append(mean_or_zero([x["goals_for"] for x in hist_b]))

    features["team_a_goals_against_avg_last5"].append(mean_or_zero([x["goals_against"] for x in hist_a]))
    features["team_b_goals_against_avg_last5"].append(mean_or_zero([x["goals_against"] for x in hist_b]))

    features["team_a_weighted_goals_for_avg_last5"].append(mean_or_zero([x["weighted_goals_for"] for x in hist_a]))
    features["team_b_weighted_goals_for_avg_last5"].append(mean_or_zero([x["weighted_goals_for"] for x in hist_b]))

    features["team_a_weighted_goals_against_avg_last5"].append(mean_or_zero([x["weighted_goals_against"] for x in hist_a]))
    features["team_b_weighted_goals_against_avg_last5"].append(mean_or_zero([x["weighted_goals_against"] for x in hist_b]))

    features["team_a_avg_opponent_elo_last5"].append(mean_or_zero([x["opponent_elo"] for x in hist_a]))
    features["team_b_avg_opponent_elo_last5"].append(mean_or_zero([x["opponent_elo"] for x in hist_b]))

    features["team_a_rating_change_avg_last5"].append(mean_or_zero([x["rating_change"] for x in hist_a]))
    features["team_b_rating_change_avg_last5"].append(mean_or_zero([x["rating_change"] for x in hist_b]))

    features["team_a_matches_played_before"].append(len(hist_a_all))
    features["team_b_matches_played_before"].append(len(hist_b_all))

    features["team_a_days_since_last_match"].append((date - hist_a_all[-1]["date"]).days if hist_a_all else 999)
    features["team_b_days_since_last_match"].append((date - hist_b_all[-1]["date"]).days if hist_b_all else 999)

    features["team_a_win_streak"].append(current_win_streak(hist_a_all))
    features["team_b_win_streak"].append(current_win_streak(hist_b_all))

    opponent_weight_for_a = row["rating_b_before"] / ELO_BASE
    opponent_weight_for_b = row["rating_a_before"] / ELO_BASE

    team_history[team_a].append({
        "date": date,
        "points": row["result_a"],
        "goals_for": row["goals_a"],
        "goals_against": row["goals_b"],
        "weighted_goals_for": row["goals_a"] * opponent_weight_for_a,
        "weighted_goals_against": row["goals_b"] * (ELO_BASE / row["rating_b_before"]),
        "opponent_elo": row["rating_b_before"],
        "rating_change": row["rating_change_a"],
    })

    team_history[team_b].append({
        "date": date,
        "points": row["result_b"],
        "goals_for": row["goals_b"],
        "goals_against": row["goals_a"],
        "weighted_goals_for": row["goals_b"] * opponent_weight_for_b,
        "weighted_goals_against": row["goals_a"] * (ELO_BASE / row["rating_a_before"]),
        "opponent_elo": row["rating_a_before"],
        "rating_change": row["rating_change_b"],
    })

for col, values in features.items():
    df[col] = values

print("Rolling features created.")
print("ELO_BASE:", round(ELO_BASE, 2))

Rolling features created.
ELO_BASE: 1482.35


## 5. Tournament State Features

In [6]:
df["tournament_year"] = df["date"].dt.year
df["tournament_key"] = df["competition"].astype(str) + "_" + df["tournament_year"].astype(str)

tournament_states = {}

tournament_features = {
    "team_a_tournament_matches_played": [],
    "team_b_tournament_matches_played": [],
    "team_a_tournament_points": [],
    "team_b_tournament_points": [],
    "tournament_points_diff": [],
    "team_a_tournament_goal_diff": [],
    "team_b_tournament_goal_diff": [],
    "tournament_goal_diff_diff": [],
    "team_a_tournament_clean_sheets": [],
    "team_b_tournament_clean_sheets": [],
    "tournament_clean_sheets_diff": [],
}

def empty_state():
    return {
        "matches": 0,
        "points": 0,
        "goals_for": 0,
        "goals_against": 0,
        "goal_diff": 0,
        "clean_sheets": 0,
    }

for _, row in df.iterrows():
    key = row["tournament_key"]
    team_a = row["team_a"]
    team_b = row["team_b"]

    tournament_states.setdefault(key, {})
    tournament_states[key].setdefault(team_a, empty_state())
    tournament_states[key].setdefault(team_b, empty_state())

    state_a = tournament_states[key][team_a]
    state_b = tournament_states[key][team_b]

    tournament_features["team_a_tournament_matches_played"].append(state_a["matches"])
    tournament_features["team_b_tournament_matches_played"].append(state_b["matches"])
    tournament_features["team_a_tournament_points"].append(state_a["points"])
    tournament_features["team_b_tournament_points"].append(state_b["points"])
    tournament_features["tournament_points_diff"].append(state_a["points"] - state_b["points"])
    tournament_features["team_a_tournament_goal_diff"].append(state_a["goal_diff"])
    tournament_features["team_b_tournament_goal_diff"].append(state_b["goal_diff"])
    tournament_features["tournament_goal_diff_diff"].append(state_a["goal_diff"] - state_b["goal_diff"])
    tournament_features["team_a_tournament_clean_sheets"].append(state_a["clean_sheets"])
    tournament_features["team_b_tournament_clean_sheets"].append(state_b["clean_sheets"])
    tournament_features["tournament_clean_sheets_diff"].append(state_a["clean_sheets"] - state_b["clean_sheets"])

    state_a["matches"] += 1
    state_a["points"] += row["result_a"]
    state_a["goals_for"] += row["goals_a"]
    state_a["goals_against"] += row["goals_b"]
    state_a["goal_diff"] = state_a["goals_for"] - state_a["goals_against"]
    state_a["clean_sheets"] += int(row["goals_b"] == 0)

    state_b["matches"] += 1
    state_b["points"] += row["result_b"]
    state_b["goals_for"] += row["goals_b"]
    state_b["goals_against"] += row["goals_a"]
    state_b["goal_diff"] = state_b["goals_for"] - state_b["goals_against"]
    state_b["clean_sheets"] += int(row["goals_a"] == 0)

for col, values in tournament_features.items():
    df[col] = values

print("Tournament state features created.")
display(df[[
    "date", "competition", "team_a", "team_b",
    "team_a_tournament_matches_played",
    "team_b_tournament_matches_played",
    "team_a_tournament_points",
    "team_b_tournament_points",
    "tournament_points_diff"
]].head(12))

Tournament state features created.


,date,competition,team_a,team_b,team_a_tournament_matches_played,team_b_tournament_matches_played,team_a_tournament_points,team_b_tournament_points,tournament_points_diff
0,2001-01-04,Friendly,Kenya,Zambia,0,0,0,0,0
1,2001-01-06,World Cup qualifier,Costa Rica,Guatemala,0,0,0,0,0
2,2001-01-06,Friendly,Egypt,United Arab Emirates,0,0,0,0,0
3,2001-01-06,Friendly tournament,Namibia,Lesotho,0,0,0,0,0
4,2001-01-06,Friendly,Qatar,Jordan,0,0,0,0,0
5,2001-01-06,Friendly tournament,Swaziland,Angola,0,0,0,0,0
6,2001-01-07,Friendly tournament,Angola,Lesotho,1,1,0,0,0
7,2001-01-07,Friendly tournament,Swaziland,Namibia,1,1,3,3,0
8,2001-01-07,Friendly,Togo,Burkina Faso,0,0,0,0,0
9,2001-01-09,Friendly,Egypt,Zambia,1,1,3,0,3


## 6. Final Focused Feature Table

In [7]:
df["form_diff_last5"] = df["team_a_form_last5"] - df["team_b_form_last5"]
df["goals_for_diff_last5"] = df["team_a_goals_for_avg_last5"] - df["team_b_goals_for_avg_last5"]
df["goals_against_diff_last5"] = df["team_a_goals_against_avg_last5"] - df["team_b_goals_against_avg_last5"]

df["weighted_goals_for_diff_last5"] = (
    df["team_a_weighted_goals_for_avg_last5"] -
    df["team_b_weighted_goals_for_avg_last5"]
)

df["weighted_goals_against_diff_last5"] = (
    df["team_a_weighted_goals_against_avg_last5"] -
    df["team_b_weighted_goals_against_avg_last5"]
)

df["opponent_strength_diff_last5"] = (
    df["team_a_avg_opponent_elo_last5"] -
    df["team_b_avg_opponent_elo_last5"]
)

df["rating_change_diff_last5"] = (
    df["team_a_rating_change_avg_last5"] -
    df["team_b_rating_change_avg_last5"]
)

df["days_since_match_diff"] = (
    df["team_a_days_since_last_match"] -
    df["team_b_days_since_last_match"]
)

df["win_streak_diff"] = df["team_a_win_streak"] - df["team_b_win_streak"]

FEATURE_COLS = [
    "rating_a_before",
    "rating_b_before",
    "elo_diff",
    "rank_diff",

    "form_diff_last5",
    "goals_for_diff_last5",
    "goals_against_diff_last5",

    "weighted_goals_for_diff_last5",
    "weighted_goals_against_diff_last5",
    "opponent_strength_diff_last5",

    "rating_change_diff_last5",
    "win_streak_diff",

    "team_a_matches_played_before",
    "team_b_matches_played_before",

    "team_a_days_since_last_match",
    "team_b_days_since_last_match",
    "days_since_match_diff",

    "team_a_tournament_matches_played",
    "team_b_tournament_matches_played",
    "tournament_points_diff",
    "tournament_goal_diff_diff",
    "tournament_clean_sheets_diff",

    "is_home_adv",
]

TARGET_COLS = ["target_goals_a", "target_goals_b"]

metadata_cols = [
    "date", "team_a", "team_b", "competition", "location",
    "tournament_year", "tournament_key"
]

model_df = df[metadata_cols + FEATURE_COLS + TARGET_COLS + ["target_goal_diff", "target_total_goals"]].copy()

model_df["team_a_days_since_last_match"] = model_df["team_a_days_since_last_match"].clip(0, 60)
model_df["team_b_days_since_last_match"] = model_df["team_b_days_since_last_match"].clip(0, 60)
model_df["days_since_match_diff"] = model_df["days_since_match_diff"].clip(-60, 60)

output_path = PROCESSED_DIR / "model_dataset.csv"
model_df.to_csv(output_path, index=False)

print("Final model dataset shape:", model_df.shape)
print("Number of features:", len(FEATURE_COLS))
print("Saved to:", output_path)
display(model_df.head())

Final model dataset shape: (24260, 34)
Number of features: 23
Saved to: ../data/processed/model_dataset.csv


,date,team_a,team_b,competition,location,tournament_year,tournament_key,rating_a_before,rating_b_before,elo_diff,rank_diff,form_diff_last5,goals_for_diff_last5,goals_against_diff_last5,weighted_goals_for_diff_last5,weighted_goals_against_diff_last5,opponent_strength_diff_last5,rating_change_diff_last5,win_streak_diff,team_a_matches_played_before,team_b_matches_played_before,team_a_days_since_last_match,team_b_days_since_last_match,days_since_match_diff,team_a_tournament_matches_played,team_b_tournament_matches_played,tournament_points_diff,tournament_goal_diff_diff,tournament_clean_sheets_diff,is_home_adv,target_goals_a,target_goals_b,target_goal_diff,target_total_goals
0,2001-01-04,Kenya,Zambia,Friendly,Kenya,2001,Friendly_2001,1328,1455,-127,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,60,60,0,0,0,0,0,0,1,2,1,1,3
1,2001-01-06,Costa Rica,Guatemala,World Cup qualifier,the United States,2001,World Cup qualifier_2001,1606,1575,31,-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,60,60,0,0,0,0,0,0,0,5,2,3,7
2,2001-01-06,Egypt,United Arab Emirates,Friendly,Egypt,2001,Friendly_2001,1682,1546,136,-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,60,60,0,0,0,0,0,0,1,2,1,1,3
3,2001-01-06,Namibia,Lesotho,Friendly tournament,Swaziland,2001,Friendly tournament_2001,1326,1196,130,-44,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,60,60,0,0,0,0,0,0,0,1,0,1,1
4,2001-01-06,Qatar,Jordan,Friendly,Qatar,2001,Friendly_2001,1495,1469,26,-8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,60,60,0,0,0,0,0,0,1,3,1,2,4


## 7. Dataset Validation

In [8]:
missing_values = model_df[FEATURE_COLS + TARGET_COLS].isna().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) > 0:
    display(missing_values)
    raise ValueError("Missing values found in model features or targets.")

non_numeric_features = [
    col for col in FEATURE_COLS
    if not pd.api.types.is_numeric_dtype(model_df[col])
]

if non_numeric_features:
    raise TypeError(f"Non-numeric feature columns found: {non_numeric_features}")

if (model_df[TARGET_COLS] < 0).any().any():
    raise ValueError("Negative goals found in target columns.")

print("Validation passed.")
print("Rows:", len(model_df))
print("Features:", len(FEATURE_COLS))
print("Date range:", model_df["date"].min().date(), "to", model_df["date"].max().date())

Validation passed.
Rows: 24260
Features: 23
Date range: 2001-01-04 to 2026-05-16


## 8. Expanding-Window World Cup Backtest

In [9]:
model_df["is_world_cup"] = model_df["competition"].astype(str).str.lower().eq("world cup").astype(int)

world_cup_years = sorted(
    model_df.loc[model_df["is_world_cup"] == 1, "date"].dt.year.unique()
)

print("World Cup years found:", world_cup_years)

World Cup years found: [np.int32(2002), np.int32(2006), np.int32(2010), np.int32(2014), np.int32(2018), np.int32(2022)]


## 9. Poisson Score Conversion

In [10]:
def poisson_prob(lam, k):
    return (lam ** k) * exp(-lam) / factorial(k)

def most_likely_score(lambda_a, lambda_b, max_goals=6):
    lambda_a = min(max(float(lambda_a), 0.05), 4.0)
    lambda_b = min(max(float(lambda_b), 0.05), 4.0)

    best_score = None
    best_prob = -1

    for ga in range(max_goals + 1):
        for gb in range(max_goals + 1):
            prob = poisson_prob(lambda_a, ga) * poisson_prob(lambda_b, gb)
            if prob > best_prob:
                best_prob = prob
                best_score = (ga, gb)

    return best_score

## 10. Run World Cup Backtest

In [11]:
results = []

for year in world_cup_years:
    wc_df = model_df[
        (model_df["is_world_cup"] == 1) &
        (model_df["date"].dt.year == year)
    ].copy()

    wc_start = wc_df["date"].min()

    train_df = model_df[
        (model_df["date"] < wc_start) &
        (model_df["rating_a_before"] >= 1420) &
        (model_df["rating_b_before"] >= 1420)
    ].copy()

    test_df = wc_df.copy()

    X_train = train_df[FEATURE_COLS]
    y_train = train_df[TARGET_COLS]

    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_COLS]

    model = MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=500,
            max_depth=10,
            min_samples_leaf=8,
            random_state=42,
            n_jobs=-1
        )
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)

    pred_scores = [
        most_likely_score(preds[i][0], preds[i][1])
        for i in range(len(preds))
    ]

    actual_scores = [
        (int(y_test.iloc[i]["target_goals_a"]), int(y_test.iloc[i]["target_goals_b"]))
        for i in range(len(y_test))
    ]

    exact_acc = np.mean([
        pred_scores[i] == actual_scores[i]
        for i in range(len(actual_scores))
    ])

    results.append({
        "world_cup_year": int(year),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "rmse": rmse,
        "mae": mae,
        "exact_score_accuracy": exact_acc,
        "exact_score_accuracy_%": round(exact_acc * 100, 2)
    })

results_df = pd.DataFrame(results)
display(results_df)

,world_cup_year,train_rows,test_rows,rmse,mae,exact_score_accuracy,exact_score_accuracy_%
0,2002,707,64,0.944816,0.695991,0.250000,25.00
1,2006,2557,64,0.878155,0.714518,0.218750,21.88
2,2010,4462,64,0.970913,0.761864,0.140625,14.06
3,2014,6428,64,1.029010,0.755662,0.093750,9.38
4,2018,8234,64,0.910102,0.701555,0.218750,21.88
5,2022,10031,64,0.963057,0.722076,0.328125,32.81


## 11. Feature Importance

In [12]:
importances_a = model.estimators_[0].feature_importances_
importances_b = model.estimators_[1].feature_importances_

importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_goals_a": importances_a,
    "importance_goals_b": importances_b,
})

importance_df["avg_importance"] = (
    importance_df["importance_goals_a"] +
    importance_df["importance_goals_b"]
) / 2

importance_df = importance_df.sort_values("avg_importance", ascending=False)

display(importance_df.head(25))

,feature,importance_goals_a,importance_goals_b,avg_importance
3,rank_diff,0.489480,0.475801,0.482641
2,elo_diff,0.145249,0.138985,0.142117
0,rating_a_before,0.046115,0.108840,0.077477
1,rating_b_before,0.069470,0.036966,0.053218
9,opponent_strength_diff_last5,0.024373,0.025763,0.025068
13,team_b_matches_played_before,0.024813,0.023200,0.024006
7,weighted_goals_for_diff_last5,0.023088,0.021498,0.022293
12,team_a_matches_played_before,0.021380,0.022386,0.021883
8,weighted_goals_against_diff_last5,0.020749,0.019848,0.020298
10,rating_change_diff_last5,0.019451,0.019231,0.019341


## 12. Save Evaluation Outputs

In [13]:
results_path = OUTPUTS_DIR / "world_cup_backtest_results.csv"
importance_path = OUTPUTS_DIR / "feature_importance.csv"

results_df.to_csv(results_path, index=False)
importance_df.to_csv(importance_path, index=False)

print("Saved:", results_path)
print("Saved:", importance_path)

Saved: ../outputs/evaluation/world_cup_backtest_results.csv
Saved: ../outputs/evaluation/feature_importance.csv


## Summary

Final dataset:

`data/processed/model_dataset.csv`

Evaluation outputs:

- `outputs/evaluation/world_cup_backtest_results.csv`
- `outputs/evaluation/feature_importance.csv`

This notebook is ready as the feature-engineering handoff to the modeling side.